# Finetune Progressive S2 — Custom Drone + NTUT 4K

Base: **progressive_stream2/seed_777** (mAP@0.5=0.5905, mAP@.5:.95=0.2004)

Data: Roboflow custom + NTUT 4K drone (pseudo-thermal).


In [13]:
import os, gc, json, math, time, random, copy
import numpy as np
import pandas as pd
import cv2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from ultralytics import YOLO
from ultralytics.utils.loss import v8DetectionLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [14]:
# =====================================================================
# Ket qua evaluation mid-stage (mAP@0.5 / mAP@.5:.95):
#   Progressive S2  seed_777: 0.5905 / 0.2004  <- BEST single ckpt
#   Progressive S2  mean:     0.5773 / 0.1955  <- BEST method
#   Progressive S1  mean:     0.5670 / 0.1886
#   ProgQFDet S1    mean:     0.5594 / 0.1856
#   ProgQFDet S2    mean:     0.5529 / 0.1853
#   Baseline S1     mean:     0.5453 / 0.1841
#   Baseline S2     mean:     0.5276 / 0.1762
#   QFDet S2        mean:     0.5186 / 0.1714
#   QFDet S1        mean:     0.5108 / 0.1648
# =====================================================================

BASE_DIR     = '/root/AIP491'
CUSTOM_DIR   = f'{BASE_DIR}/custom_data'
NTUT_DIR     = f'{CUSTOM_DIR}/NTUT'
BACKBONE_DIR = f'{BASE_DIR}/backbones'
MID_OUT_DIR  = f'{BASE_DIR}/Mid-stage/outputs'

# Best checkpoint: Progressive Stream2 seed_777
PRETRAIN_CKPT = f'{MID_OUT_DIR}/progressive_stream2/seed_777/fusion_best.pt'
RGB_BB_PATH   = f'{BACKBONE_DIR}/llvip_rgb_best.pt'    # Stream 2: LLVIP RGB
THR_BB_PATH   = f'{BACKBONE_DIR}/llvip_thermal_best.pt'

# Finetune hyperparams
IMG_SIZE         = 640
BATCH_SIZE       = 16
NUM_WORKERS      = 4
GRAD_CLIP        = 1.0
SEEDS            = [42, 777, 123]
CUSTOM_VAL_RATIO = 0.2

# LR thap hon training goc ~5x de tranh catastrophic forgetting
# Progressive S2 tune params: stage1=7.7e-6, stage2=1.2e-4, stage3=8.0e-4
# Finetune: chia 5 -> stage1~2e-6, stage2~2e-5, stage3~1.5e-4
STAGE1_LR     = 2e-6    # freeze backbone, chi train fusion+neck+head
STAGE1_EPOCHS = 5
STAGE2_LR     = 2e-5    # unfreeze 3 layers cuoi backbone
STAGE2_EPOCHS = 5
STAGE3_LR     = 1.5e-4  # full unfreeze + cosine decay
STAGE3_MAX_EP = 20
PATIENCE      = 3

print(f'Checkpoint: progressive_stream2/seed_777')
print(f'Backbone:   LLVIP(RGB) + LLVIP(Thermal)  [Stream 2]')
print(f'Batch: {BATCH_SIZE} | Seeds: {SEEDS}')


Checkpoint: progressive_stream2/seed_777
Backbone:   LLVIP(RGB) + LLVIP(Thermal)  [Stream 2]
Batch: 16 | Seeds: [42, 777, 123]


In [15]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


In [16]:
# =====================================================================
# Parse Custom Drone Data (Roboflow COCO format)
# =====================================================================
def parse_custom_coco(custom_dir, val_ratio=0.2, seed=42):
    ann_path = Path(custom_dir) / 'annotation.json'
    rgb_dir  = Path(custom_dir) / 'rgb'
    thr_dir  = Path(custom_dir) / 'thermal_1'

    with open(str(ann_path)) as f:
        coco = json.load(f)

    id2img = {img['id']: (img['file_name'],
                          float(img.get('width', 1920)),
                          float(img.get('height', 1080)))
              for img in coco['images']}

    valid_cats = {cat['id']: 0 for cat in coco['categories']
                  if any(k in cat['name'].lower() for k in ('person', 'people'))}
    if not valid_cats:
        valid_cats = {1: 0}

    img2boxes = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        if ann['category_id'] not in valid_cats or img_id not in id2img:
            continue
        _, iw, ih = id2img[img_id]
        x, y, bw, bh = [float(v) for v in ann['bbox']]   # COCO: xmin,ymin,w,h
        cx, cy = (x + bw / 2.0) / iw, (y + bh / 2.0) / ih
        nw, nh = bw / iw, bh / ih
        if nw > 0 and nh > 0:
            img2boxes.setdefault(img_id, []).append([0, cx, cy, nw, nh])

    all_samples = []
    for img_id, boxes in img2boxes.items():
        fname = id2img[img_id][0]
        rgb_p = rgb_dir / fname
        thr_p = thr_dir / fname
        if not rgb_p.exists():
            continue
        all_samples.append({
            'rgb_path':   str(rgb_p),
            'thr_path':   str(thr_p) if thr_p.exists() else None,
            'pseudo_thr': not thr_p.exists(),
            'boxes':      boxes,
            'source':     'custom',
        })

    rng = random.Random(seed)
    rng.shuffle(all_samples)
    n_val = max(1, int(len(all_samples) * val_ratio))
    val_s, train_s = all_samples[:n_val], all_samples[n_val:]
    n_real_thr = sum(1 for s in all_samples if not s['pseudo_thr'])
    print(f'Custom: {len(train_s)} train | {len(val_s)} val')
    print(f'  thermal_1 co san: {n_real_thr}/{len(all_samples)} anh')
    return train_s, val_s


custom_train, custom_val = parse_custom_coco(CUSTOM_DIR, CUSTOM_VAL_RATIO)


Custom: 1328 train | 331 val
  thermal_1 co san: 1659/1659 anh


In [17]:
# =====================================================================
# Parse NTUT Data (VoTT CSV, RGB-only -> pseudo-thermal)
# =====================================================================
# CSV toa do la goc 4K (3840x2160), anh tren disk da resize ve 720p
# -> normalize dung kich thuoc goc, KHONG doc tu anh da resize
NTUT_ORIG_W = 3840
NTUT_ORIG_H = 2160

def parse_ntut_split(ntut_dir, split_folder):
    # Tat ca label trong NTUT deu la nguoi -> map tat ca ve class 0
    samples = []
    split_root = Path(ntut_dir) / split_folder / split_folder
    if not split_root.exists():
        print(f'  [WARN] NTUT not found: {split_root}')
        return samples

    for drone_dir in sorted(split_root.iterdir()):
        if not drone_dir.is_dir():
            continue
        csv_dir  = drone_dir / 'vott-csv-export'
        csv_path = csv_dir / f'{drone_dir.name}-export.csv'
        if not csv_path.exists():
            continue

        df = pd.read_csv(str(csv_path))
        groups = {}
        for _, row in df.iterrows():
            img_name = str(row['image']).strip().strip('"')
            if not img_name.lower().endswith('.jpg'):
                img_name += '.jpg'
            img_path = csv_dir / img_name
            if not img_path.exists():
                continue
            groups.setdefault(img_path, []).append(
                (0, float(row['xmin']), float(row['ymin']),
                    float(row['xmax']), float(row['ymax'])))

        n_before = len(samples)
        for img_path, raw_boxes in groups.items():
            boxes = []
            for cls_id, xmin, ymin, xmax, ymax in raw_boxes:
                xmin = max(0.0, xmin)
                ymin = max(0.0, ymin)
                xmax = min(float(NTUT_ORIG_W), xmax)
                ymax = min(float(NTUT_ORIG_H), ymax)
                if xmax > xmin and ymax > ymin:
                    boxes.append([cls_id,
                                  (xmin+xmax)/2/NTUT_ORIG_W,
                                  (ymin+ymax)/2/NTUT_ORIG_H,
                                  (xmax-xmin)/NTUT_ORIG_W,
                                  (ymax-ymin)/NTUT_ORIG_H])
            if boxes:
                samples.append({
                    'rgb_path':   str(img_path),
                    'thr_path':   None,
                    'pseudo_thr': True,
                    'boxes':      boxes,
                    'source':     'ntut',
                })
        print(f'  {drone_dir.name}: {len(samples)-n_before} mau')
    return samples


ntut_train = parse_ntut_split(NTUT_DIR, 'ntut_drone_train')
ntut_val   = parse_ntut_split(NTUT_DIR, 'ntut_drone_test')
print(f'NTUT: {len(ntut_train)} train | {len(ntut_val)} val')


  Drone_005: 740 mau
  Drone_023: 541 mau
  Drone_031: 485 mau
  Drone_049: 371 mau
  Drone_004: 506 mau
  Drone_014: 720 mau
  Drone_024: 281 mau
  Drone_050: 425 mau
NTUT: 2137 train | 1932 val


In [18]:
# NTUT toan bo dung cho train (pseudo-thermal, khong co real thermal de val)
# Val chi giu custom drone co real thermal pairs
train_samples = custom_train + ntut_train + ntut_val
val_samples   = custom_val
print(f'Train: {len(train_samples)} (custom={len(custom_train)}, ntut={len(ntut_train)+len(ntut_val)})')
print(f'Val:   {len(val_samples)} (custom only, real thermal)')


Train: 5397 (custom=1328, ntut=4069)
Val:   331 (custom only, real thermal)


In [19]:
# =====================================================================
# CombinedRGBTDataset  (with augmentation: flip, zoom, blur, color)
# =====================================================================
class CombinedRGBTDataset(Dataset):
    def __init__(self, samples, img_size=640, is_train=True):
        self.samples   = samples
        self.img_size  = img_size
        self.is_train  = is_train
        self.resize    = T.Resize((img_size, img_size))
        self.to_tensor = T.ToTensor()
        src = {}
        for s in samples:
            src[s['source']] = src.get(s['source'], 0) + 1
        split_name = 'train' if is_train else 'val'
        print(f'Dataset [{split_name}]: {len(samples)} mau | {src}')

    def __len__(self): return len(self.samples)

    def _pseudo_thermal(self, rgb_pil):
        arr  = np.array(rgb_pil)
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        hot  = cv2.applyColorMap(gray, cv2.COLORMAP_HOT)
        return Image.fromarray(cv2.cvtColor(hot, cv2.COLOR_BGR2RGB))

    def _random_zoom(self, rgb_pil, thr_pil, boxes):
        '''Zoom in/out dong thoi tren ca RGB va Thermal, cap nhat boxes.'''
        scale = random.uniform(0.75, 1.25)
        sz    = self.img_size
        if scale > 1.0:
            # Zoom in: crop center roi resize ve sz
            crop_sz = int(sz / scale)
            x1 = (sz - crop_sz) // 2
            y1 = (sz - crop_sz) // 2
            rgb_pil = rgb_pil.crop((x1, y1, x1 + crop_sz, y1 + crop_sz)).resize((sz, sz))
            thr_pil = thr_pil.crop((x1, y1, x1 + crop_sz, y1 + crop_sz)).resize((sz, sz))
            # Box: tu toa do normalized -> absolute crop -> normalized moi
            new_boxes = []
            for b in boxes:
                cls, cx, cy, bw, bh = b
                ax, ay = cx * sz - x1, cy * sz - y1
                aw, ah = bw * sz, bh * sz
                xmin, ymin = max(ax - aw / 2, 0), max(ay - ah / 2, 0)
                xmax, ymax = min(ax + aw / 2, crop_sz), min(ay + ah / 2, crop_sz)
                if xmax > xmin and ymax > ymin:
                    ncx, ncy = (xmin + xmax) / 2 / crop_sz, (ymin + ymax) / 2 / crop_sz
                    nbw, nbh = (xmax - xmin) / crop_sz, (ymax - ymin) / crop_sz
                    if nbw > 0.01 and nbh > 0.01:
                        new_boxes.append([cls, ncx, ncy, nbw, nbh])
            boxes = new_boxes if new_boxes else boxes
        else:
            # Zoom out: resize nho roi pad black
            new_sz = int(sz * scale)
            x1 = (sz - new_sz) // 2
            y1 = (sz - new_sz) // 2
            rgb_small = rgb_pil.resize((new_sz, new_sz))
            thr_small = thr_pil.resize((new_sz, new_sz))
            rgb_out = Image.new('RGB', (sz, sz), (0, 0, 0))
            thr_out = Image.new('RGB', (sz, sz), (0, 0, 0))
            rgb_out.paste(rgb_small, (x1, y1))
            thr_out.paste(thr_small, (x1, y1))
            rgb_pil, thr_pil = rgb_out, thr_out
            boxes = [[b[0],
                      (b[1] * new_sz + x1) / sz,
                      (b[2] * new_sz + y1) / sz,
                      b[3] * new_sz / sz,
                      b[4] * new_sz / sz] for b in boxes]
        return rgb_pil, thr_pil, boxes

    def __getitem__(self, idx):
        s = self.samples[idx]
        rgb_pil = Image.open(s['rgb_path']).convert('RGB')
        if s['pseudo_thr'] or s['thr_path'] is None:
            thr_pil = self._pseudo_thermal(rgb_pil)
        else:
            thr_pil = Image.open(s['thr_path']).convert('L').convert('RGB')

        rgb_pil = self.resize(rgb_pil)
        thr_pil = self.resize(thr_pil)
        boxes   = [b[:] for b in s['boxes']]

        if self.is_train:
            # Horizontal flip
            if random.random() < 0.5:
                rgb_pil = TF.hflip(rgb_pil)
                thr_pil = TF.hflip(thr_pil)
                boxes   = [[b[0], 1 - b[1], b[2], b[3], b[4]] for b in boxes]
            # Vertical flip (drone bay nhieu goc)
            if random.random() < 0.2:
                rgb_pil = TF.vflip(rgb_pil)
                thr_pil = TF.vflip(thr_pil)
                boxes   = [[b[0], b[1], 1 - b[2], b[3], b[4]] for b in boxes]
            # Random zoom (simulate altitude change)
            if random.random() < 0.5:
                rgb_pil, thr_pil, boxes = self._random_zoom(rgb_pil, thr_pil, boxes)
            # Gaussian blur (motion blur / out-of-focus)
            if random.random() < 0.3:
                k = random.choice([3, 5])
                rgb_pil = Image.fromarray(cv2.GaussianBlur(np.array(rgb_pil), (k, k), 0))
                thr_pil = Image.fromarray(cv2.GaussianBlur(np.array(thr_pil), (k, k), 0))
            # Color jitter chi tren RGB (thermal khong doi)
            if random.random() < 0.4:
                rgb_pil = TF.adjust_brightness(rgb_pil, random.uniform(0.6, 1.4))
                rgb_pil = TF.adjust_contrast(rgb_pil,   random.uniform(0.7, 1.3))
                rgb_pil = TF.adjust_saturation(rgb_pil, random.uniform(0.5, 1.5))
                rgb_pil = TF.adjust_hue(rgb_pil,        random.uniform(-0.1, 0.1))

        rgb_t = self.to_tensor(rgb_pil)
        thr_t = self.to_tensor(thr_pil)
        lbl_t = (torch.tensor(boxes, dtype=torch.float32)
                 if boxes else torch.zeros((0, 5)))
        return rgb_t, thr_t, lbl_t, Path(s['rgb_path']).stem


def collate_fn(batch):
    rgbs, thrs, lbls, stems = zip(*batch)
    rgbs = torch.stack(rgbs)
    thrs = torch.stack(thrs)
    out  = [torch.cat([torch.full((len(l), 1), float(i)), l], dim=1)
            for i, l in enumerate(lbls) if l.shape[0] > 0]
    return rgbs, thrs, (torch.cat(out) if out else torch.zeros((0, 6))), list(stems)


In [20]:
# =====================================================================
# Progressive Fusion (Stream 2: LLVIP_RGB + LLVIP_Thermal)
# Import Conv/C2f/Detect truc tiep tu ultralytics de dam bao khop
# state_dict voi checkpoint mid-stage (ultralytics Detect dung DWConv)
# =====================================================================
from ultralytics.nn.modules import C2f, Conv, Detect


class RGBTFusionDetector(nn.Module):
    '''Progressive Mid-Fusion (khop voi checkpoint progressive_stream2).'''
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}

    def __init__(self, rgb_backbone, thermal_backbone, nc=1, freeze_backbones=True):
        super().__init__()
        self.rgb_stream     = rgb_backbone
        self.thermal_stream = thermal_backbone
        if freeze_backbones:
            for p in (list(self.rgb_stream.parameters()) +
                      list(self.thermal_stream.parameters())):
                p.requires_grad = False

        self.fuse_p3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.SiLU())
        self.fuse_p4 = nn.Sequential(
            nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU())
        self.fuse_p5 = nn.Sequential(
            nn.Conv2d(512, 512, 1, bias=False), nn.BatchNorm2d(512), nn.SiLU())
        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(512 + 256, 256, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(256 + 128, 128, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(128, 128, 3, 2)
        self.bu_c2f_p4  = C2f(128 + 256, 256, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(256, 256, 3, 2)
        self.bu_c2f_p5  = C2f(256 + 512, 512, n=1, shortcut=False)
        self.detect     = Detect(nc=nc, ch=(128, 256, 512))
        self.detect.stride = torch.tensor([8., 16., 32.])

    def unfreeze_backbone_layers(self, n_layers):
        start = max(0, 10 - n_layers)
        for stream in [self.rgb_stream, self.thermal_stream]:
            for i, layer in enumerate(stream):
                if i >= start:
                    for p in layer.parameters():
                        p.requires_grad = True

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

    def _extract(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS:
                feats[i] = x
        return feats

    def forward(self, rgb, thermal):
        rf = self._extract(self.rgb_stream, rgb)
        tf = self._extract(self.thermal_stream, thermal)
        p3 = self.fuse_p3(torch.cat([rf[4], tf[4]], dim=1))
        p4 = self.fuse_p4(torch.cat([rf[6], tf[6]], dim=1))
        p5 = self.fuse_p5(torch.cat([rf[9], tf[9]], dim=1))
        p4_td  = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], dim=1))
        p3_out = self.td_c2f_p3(torch.cat([self.upsample(p4_td), p3], dim=1))
        p4_out = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3_out), p4_td], dim=1))
        p5_out = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4_out), p5], dim=1))
        return self.detect([p3_out, p4_out, p5_out])


def load_backbone(path):
    src = path if (path and os.path.exists(path)) else 'yolov8n.pt'
    return nn.ModuleList(list(YOLO(src).model.model)[:10]).to(device)


def build_model():
    rgb_bb = load_backbone(RGB_BB_PATH)
    thr_bb = load_backbone(THR_BB_PATH)
    model  = RGBTFusionDetector(rgb_bb, thr_bb, nc=1,
                                freeze_backbones=True).to(device)
    ckpt   = torch.load(PRETRAIN_CKPT, map_location=device, weights_only=False)
    state  = ckpt.get('model_state_dict', ckpt)
    miss, unexp = model.load_state_dict(state, strict=False)
    if miss:  print(f'  [WARN] missing:    {miss[:3]}')
    if unexp: print(f'  [WARN] unexpected: {unexp[:3]}')
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f'  Loaded progressive_stream2/seed_777')
    print(f'  Params trainable: {n_trainable:,} / {n_total:,}')
    return model


print('Model class OK.')


Model class OK.


In [21]:
# =====================================================================
# Loss + run_epoch
# =====================================================================
class LossModel(nn.Module):
    def __init__(self, detector):
        super().__init__()
        self.model = nn.ModuleList([detector.detect])
        self.nc    = detector.detect.nc
        self.args  = type('a', (), {
            'box': 7.5, 'cls': 0.5, 'dfl': 1.5,
            'pose': 12.0, 'kobj': 1.0
        })()


def run_epoch(model, loader, optimizer, criterion, train=True):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for i, (rgb, thr, labels, _) in enumerate(loader):
            rgb    = rgb.to(device)
            thr    = thr.to(device)
            labels = labels.to(device)
            preds  = model(rgb, thr)
            bd = {'cls': labels[:, 1:2], 'bboxes': labels[:, 2:6],
                  'batch_idx': labels[:, 0], 'img': rgb}
            lr   = criterion(preds, bd)
            loss = (lr[0].sum() if isinstance(lr, (tuple, list))
                    else lr.sum()) / rgb.shape[0]
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    GRAD_CLIP)
                optimizer.step()
            total += loss.item()
            n     += 1
            if train and i % 30 == 0:
                print(f'    [{i:4d}/{len(loader)}] loss={loss.item():.4f}')
    return total / max(n, 1)


In [22]:
# =====================================================================
# Finetune 1 seed: 3 stages progressive unfreeze
# =====================================================================
def finetune_one_seed(seed, train_samples, val_samples, out_dir, tag):
    set_seed(seed)
    seed_dir = os.path.join(out_dir, f'seed_{seed}')
    os.makedirs(seed_dir, exist_ok=True)

    # Resume: skip neu da co checkpoint
    done_path = os.path.join(seed_dir, 'ft_best.pt')
    res_path  = os.path.join(seed_dir, 'ft_results.json')
    if os.path.exists(done_path) and os.path.exists(res_path):
        with open(res_path) as f:
            d = json.load(f)
        print(f'[Seed {seed}] Resume: best_val={d["best_val"]:.4f} (skip)')
        return d['best_val']

    tr_dl = DataLoader(
        CombinedRGBTDataset(train_samples, IMG_SIZE, is_train=True),
        BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        collate_fn=collate_fn, pin_memory=True)
    vl_dl = DataLoader(
        CombinedRGBTDataset(val_samples, IMG_SIZE, is_train=False),
        BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        collate_fn=collate_fn, pin_memory=True)

    model     = build_model()
    criterion = v8DetectionLoss(LossModel(model))
    hist      = {'train': [], 'val': [], 'stage': []}
    best_val  = float('inf')

    def _save_best(stage):
        torch.save({'model_state_dict': model.state_dict(),
                    'val_loss': best_val, 'seed': seed,
                    'stage': stage, 'tag': tag}, done_path)

    def _log(stage, ep, tr, vl):
        nonlocal best_val
        hist['train'].append(tr)
        hist['val'].append(vl)
        hist['stage'].append(stage)
        print(f'  Stage{stage} ep{ep}: train={tr:.4f}  val={vl:.4f}')
        if vl < best_val:
            best_val = vl
            _save_best(stage)
            print(f'    ** Best: {best_val:.4f} **')
        return vl >= best_val   # True = no improve

    # ── Stage 1: freeze backbone ──
    print(f'\n[Seed {seed}] Stage 1 ({STAGE1_EPOCHS}ep, LR={STAGE1_LR})')
    opt = AdamW([p for p in model.parameters() if p.requires_grad],
                lr=STAGE1_LR, weight_decay=1e-4)
    crit = v8DetectionLoss(LossModel(model))
    for ep in range(1, STAGE1_EPOCHS + 1):
        _log(1, ep, run_epoch(model, tr_dl, opt, crit, True),
                    run_epoch(model, vl_dl, opt, crit, False))

    # ── Stage 2: unfreeze 3 layers ──
    print(f'\n[Seed {seed}] Stage 2 ({STAGE2_EPOCHS}ep, LR={STAGE2_LR})')
    model.unfreeze_backbone_layers(3)
    opt  = AdamW([p for p in model.parameters() if p.requires_grad],
                 lr=STAGE2_LR, weight_decay=1e-4)
    crit = v8DetectionLoss(LossModel(model))
    for ep in range(1, STAGE2_EPOCHS + 1):
        _log(2, ep, run_epoch(model, tr_dl, opt, crit, True),
                    run_epoch(model, vl_dl, opt, crit, False))

    # ── Stage 3: full unfreeze + cosine + early stop ──
    print(f'\n[Seed {seed}] Stage 3 (max {STAGE3_MAX_EP}ep, LR={STAGE3_LR})')
    model.unfreeze_all()
    opt   = AdamW(model.parameters(), lr=STAGE3_LR, weight_decay=5e-5)
    sched = CosineAnnealingLR(opt, T_max=STAGE3_MAX_EP,
                              eta_min=STAGE3_LR * 0.05)
    no_imp = 0
    for ep in range(1, STAGE3_MAX_EP + 1):
        tr = run_epoch(model, tr_dl, opt, crit, True)
        vl = run_epoch(model, vl_dl, opt, crit, False)
        sched.step()
        lr_now = sched.get_last_lr()[0]
        print(f'  Stage3 ep{ep}: train={tr:.4f}  val={vl:.4f}  lr={lr_now:.2e}')
        hist['train'].append(tr); hist['val'].append(vl); hist['stage'].append(3)
        if vl < best_val:
            best_val = vl; no_imp = 0
            _save_best(3)
            print(f'    ** Best: {best_val:.4f} **')
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                print(f'  Early stop @ ep{ep}')
                break

    # Loss curve
    ep_nums = list(range(1, len(hist['train']) + 1))
    stage_changes = [i for i in range(1, len(hist['stage']))
                     if hist['stage'][i] != hist['stage'][i - 1]]
    plt.figure(figsize=(9, 4))
    plt.plot(ep_nums, hist['train'], label='Train', color='steelblue')
    plt.plot(ep_nums, hist['val'],   label='Val',   color='darkorange')
    for x in stage_changes:
        plt.axvline(x=x + 1, color='gray', linestyle='--', alpha=0.6)
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Finetune [{tag}] Seed {seed} | best_val={best_val:.4f}')
    plt.tight_layout()
    plt.savefig(os.path.join(seed_dir, 'ft_loss.png'), dpi=100)
    plt.close()

    with open(res_path, 'w') as f:
        json.dump({'seed': seed, 'best_val': best_val,
                   'epochs': len(hist['train']), 'hist': hist}, f, indent=2)

    del model, opt, crit
    gc.collect(); torch.cuda.empty_cache()
    print(f'[Seed {seed}] Done. Best val: {best_val:.4f}')
    return best_val


In [23]:
# =====================================================================
# Chay finetune 3 seeds
# =====================================================================
FT_OUT_DIR = os.path.join(BASE_DIR, 'Finetune', 'outputs', 'progressive_s2_custom_ntut')
os.makedirs(FT_OUT_DIR, exist_ok=True)
print(f'Output: {FT_OUT_DIR}')

results = {}
for seed in SEEDS:
    print('\n' + '=' * 60)
    print(f' [CUSTOM_NTUT] SEED {seed}')
    print('=' * 60)
    results[seed] = finetune_one_seed(
        seed, train_samples, val_samples, FT_OUT_DIR, tag='custom_ntut')

best_seed = min(results, key=results.get)
print('\n' + '=' * 60)
print(f'KET QUA FINETUNE [CUSTOM_NTUT]:')
for s, v in sorted(results.items(), key=lambda x: x[1]):
    mark = ' << BEST' if s == best_seed else ''
    print(f'  Seed {s}: val_loss={v:.4f}{mark}')

with open(os.path.join(FT_OUT_DIR, 'summary.json'), 'w') as f:
    json.dump({'tag': 'custom_ntut',
               'base_ckpt': 'progressive_stream2/seed_777',
               'base_map50': 0.5905, 'base_map5095': 0.2004,
               'results': results,
               'best_seed': int(best_seed),
               'best_val_loss': float(results[best_seed])}, f, indent=2)
print(f'Saved: {FT_OUT_DIR}/summary.json')


Output: /root/AIP491/Finetune/outputs/progressive_s2_custom_ntut

 [CUSTOM_NTUT] SEED 42
Dataset [train]: 5397 mau | {'custom': 1328, 'ntut': 4069}
Dataset [val]: 331 mau | {'custom': 331}
  Loaded progressive_stream2/seed_777
  Params trainable: 5,105,475 / 7,650,803

[Seed 42] Stage 1 (5ep, LR=2e-06)
    [   0/338] loss=8.5848
    [  30/338] loss=8.4195
    [  60/338] loss=8.8429
    [  90/338] loss=8.5806
    [ 120/338] loss=7.3566
    [ 150/338] loss=9.1669
    [ 180/338] loss=7.5912
    [ 210/338] loss=7.9612
    [ 240/338] loss=7.7439
    [ 270/338] loss=8.1967
    [ 300/338] loss=7.5596
    [ 330/338] loss=7.1282
  Stage1 ep1: train=8.1457  val=6.6099
    ** Best: 6.6099 **
    [   0/338] loss=6.8874
    [  30/338] loss=7.3820
    [  60/338] loss=6.4748
    [  90/338] loss=8.3377
    [ 120/338] loss=8.6480
    [ 150/338] loss=7.9015
    [ 180/338] loss=7.0381
    [ 210/338] loss=7.6573
    [ 240/338] loss=7.1184
    [ 270/338] loss=7.1992
    [ 300/338] loss=7.4635
    [ 330/338

In [24]:
# So sanh loss curves ca 3 seeds
fig, axes = plt.subplots(1, len(SEEDS), figsize=(14, 4), sharey=True)
for ax, seed in zip(axes, SEEDS):
    p = os.path.join(FT_OUT_DIR, f'seed_{seed}', 'ft_results.json')
    if not os.path.exists(p):
        continue
    with open(p) as f:
        d = json.load(f)
    ep = list(range(1, len(d['hist']['train']) + 1))
    ax.plot(ep, d['hist']['train'], label='Train', color='steelblue')
    ax.plot(ep, d['hist']['val'],   label='Val',   color='darkorange')
    ax.axhline(y=d['best_val'], color='red', linestyle=':', alpha=0.7,
               label=f'best={d["best_val"]:.4f}')
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    if seed == SEEDS[0]:
        ax.set_ylabel('Loss')
        ax.legend(fontsize=8)

name = os.path.basename(FT_OUT_DIR)
plt.suptitle(f'Finetune Progressive S2 [{name}]', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FT_OUT_DIR, 'all_seeds.png'), dpi=120)
plt.show()
print(f'Saved: {FT_OUT_DIR}/all_seeds.png')


Saved: /root/AIP491/Finetune/outputs/progressive_s2_custom_ntut/all_seeds.png


In [ ]:
# =====================================================================
# Export ONNX -- chay sau khi finetune xong
# Output: Finetune/outputs/progressive_s2_custom_ntut/fusion_finetune.onnx
# =====================================================================
import onnx

# Tim best seed tu summary.json
summary_path = os.path.join(FT_OUT_DIR, 'summary.json')
with open(summary_path) as f:
    summary = json.load(f)
best_seed = summary['best_seed']
best_ckpt = os.path.join(FT_OUT_DIR, f'seed_{best_seed}', 'ft_best.pt')
print(f'Best seed: {best_seed}  checkpoint: {best_ckpt}')

# Build model va load checkpoint
export_model = build_model()
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
export_model.load_state_dict(ckpt['model_state_dict'])
export_model.eval()

# Tat Detect postprocess khi export
export_model.detect.export = True
export_model.detect.format = 'onnx'

# Dummy inputs
dummy_rgb = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
dummy_thr = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

output_onnx = os.path.join(FT_OUT_DIR, 'fusion_finetune.onnx')
print(f'Exporting -> {output_onnx}')

torch.onnx.export(
    export_model,
    (dummy_rgb, dummy_thr),
    output_onnx,
    input_names=['rgb', 'thermal'],
    output_names=['output'],
    dynamic_axes={
        'rgb':     {0: 'batch'},
        'thermal': {0: 'batch'},
        'output':  {0: 'batch'},
    },
    opset_version=17,
    do_constant_folding=True,
)

onnx.checker.check_model(onnx.load(output_onnx))
size_mb = os.path.getsize(output_onnx) / (1024*1024)
print(f'Export OK: {output_onnx} ({size_mb:.1f} MB)')
print(f'Download: scp root@<server>:{output_onnx} Demo/models/fusion_progressive_finetune.onnx')

del export_model
gc.collect(); torch.cuda.empty_cache()
